# 04 — Gold/Blue analysis and saved figures

This notebook is CPU-only after the sweep. Every displayed plot is saved
immediately as a PNG, and its underlying table is saved as CSV. Headline
results exclude own-secret output leakage and use the predeclared layer band
37–58 at the final input token. Full layer/position plots are exploratory.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
RUN_ID = "PASTE_RUN_ID_FROM_NOTEBOOK_01"

from src.experiment_io import open_run
from src.behavior import require_behavior_approval
from src.jlens_sanity import require_sanity_approval

paths, config = open_run(RUN_ID)
require_behavior_approval(paths, config)
require_sanity_approval(paths, config)
assert (paths.result_dir / "lens_readouts.parquet").exists(), "Run notebook 03 first."
print(paths.result_dir)


## 1. Behavioral validity and leakage


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from src.behavior import behavior_dataframe
from src.experiment_io import read_jsonl

behavior = behavior_dataframe(paths)
behavior_summary = (
    behavior.groupby(["condition", "prompt_type"], as_index=False)
    .agg(
        leak_rate=("own_secret_leaked", "mean"),
        differs_from_base_rate=("differs_from_base", "mean"),
        mean_tokens=("generation_token_count", "mean"),
    )
)
behavior_long = behavior_summary.melt(
    id_vars=["condition", "prompt_type"],
    value_vars=["leak_rate", "differs_from_base_rate"],
    var_name="metric", value_name="rate",
)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=behavior_long, x="condition", y="rate", hue="metric", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("Gold/Blue behavior checks across published prompts")
saved_path = paths.figure_dir / "behavior_summary.png"
fig.savefig(saved_path, dpi=180, bbox_inches="tight")
behavior_summary.to_csv(paths.result_dir / "behavior_summary_for_plot.csv", index=False)
display(fig)
display(behavior_summary)
print("Saved:", saved_path)


## 2. Independent base-model J-Lens sanity


In [ ]:
sanity_files = sorted((paths.lens_dir / "sanity").glob("*.jsonl"))
assert sanity_files, "J-Lens sanity output is missing."
sanity = pd.DataFrame(read_jsonl(sanity_files[0]))
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(data=sanity, x="layer", y="target_rank", hue="method", marker="o", ax=ax)
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_title(f"Official base-model sanity: rank of {sanity['target'].iloc[0]!r}")
saved_path = paths.figure_dir / "base_jlens_sanity_rank.png"
fig.savefig(saved_path, dpi=180, bbox_inches="tight")
sanity.to_csv(paths.result_dir / "base_jlens_sanity.csv", index=False)
display(fig)
display(sanity.sort_values("target_rank").head(20))
print("Saved:", saved_path)


## 3. Preregistered headline metrics

Candidate MRR asks whether the correct active adapter wins against the other
Gold/Blue candidate. Full-vocabulary rank asks whether the exact word is
visible without assuming a two-word answer set. `delta_*` columns are paired
J-Lens minus Logit Lens values on the same example.


In [ ]:
lens_path = paths.result_dir / "lens_readouts.parquet"
headline_columns = [
    "prompt_id", "prompt_type", "condition", "target_word", "method", "layer",
    "position_roles_json", "own_secret_leaked", "target_candidate_rank",
    "target_margin", "target_full_rank", "predicted_candidate",
]
headline_rows = pd.read_parquet(lens_path, columns=headline_columns)
headline_rows = headline_rows[headline_rows["target_word"].notna()].copy()
headline_rows["position_roles"] = headline_rows["position_roles_json"].map(json.loads)
headline_rows = headline_rows[
    headline_rows["position_roles"].map(lambda roles: "last_input" in roles)
]
band_start, band_end = config["readout"]["prior_band"]
headline_rows = headline_rows[headline_rows["layer"].between(band_start, band_end)]
if config["readout"]["exclude_leaking_outputs_from_headline"]:
    headline_rows = headline_rows[~headline_rows["own_secret_leaked"]]

# Candidate MRR assumes the known Gold/Blue set; full_hit5 does not.
headline_rows["candidate_rr"] = 1.0 / headline_rows["target_candidate_rank"]
headline_rows["candidate_hit1"] = headline_rows["target_candidate_rank"].le(1)
headline_rows["full_hit5"] = headline_rows["target_full_rank"].le(5)
per_example = (
    headline_rows.groupby(
        ["prompt_id", "prompt_type", "condition", "target_word", "method"],
        as_index=False,
    )
    .agg(
        mean_candidate_rr=("candidate_rr", "mean"),
        band_candidate_recall1=("candidate_hit1", "max"),
        mean_target_margin=("target_margin", "mean"),
        best_full_rank=("target_full_rank", "min"),
        band_full_recall5=("full_hit5", "max"),
    )
)
headline = (
    per_example.groupby(["condition", "prompt_type", "method"], as_index=False)
    .agg(
        n=("prompt_id", "size"),
        mean_candidate_mrr=("mean_candidate_rr", "mean"),
        candidate_recall1=("band_candidate_recall1", "mean"),
        mean_target_margin=("mean_target_margin", "mean"),
        median_best_full_rank=("best_full_rank", "median"),
        full_recall5=("band_full_recall5", "mean"),
    )
)
headline.to_csv(paths.result_dir / "headline_metrics.csv", index=False)

# Pair J-Lens and Logit Lens on the same prompt/condition before differencing.
paired = per_example.pivot_table(
    index=["prompt_id", "prompt_type", "condition", "target_word"],
    columns="method",
    values=["mean_candidate_rr", "mean_target_margin", "best_full_rank"],
)
paired.columns = [f"{metric}__{method}" for metric, method in paired.columns]
paired = paired.reset_index()
for metric in ("mean_candidate_rr", "mean_target_margin"):
    paired[f"delta_{metric}_j_minus_logit"] = (
        paired[f"{metric}__jlens"] - paired[f"{metric}__logit_lens"]
    )
paired["delta_best_full_rank_logit_minus_j"] = (
    paired["best_full_rank__logit_lens"] - paired["best_full_rank__jlens"]
)
paired.to_csv(paths.result_dir / "paired_method_comparison.csv", index=False)
display(headline)
display(paired)


## 4. Adapter effect relative to base

This control asks whether attaching Gold or Blue raises its own target signal
relative to the unadapted model on the identical prompt and layer. A positive
target-margin shift is adapter-specific; a high absolute score shared with
base could instead reflect vocabulary frequency or generic prompt effects.


In [ ]:
effect_columns = [
    "prompt_id", "prompt_type", "condition", "method", "layer",
    "position_roles_json", "own_secret_leaked", "gold_logit", "blue_logit",
    "gold_full_rank", "blue_full_rank",
]
effect = pd.read_parquet(lens_path, columns=effect_columns)
effect["position_roles"] = effect["position_roles_json"].map(json.loads)
effect = effect[effect["position_roles"].map(lambda roles: "last_input" in roles)]
effect = effect[effect["layer"].between(band_start, band_end)].copy()
base_effect = effect[effect["condition"].eq("base")].copy()
adapter_effect = effect[effect["condition"].isin(("gold", "blue"))].copy()
if config["readout"]["exclude_leaking_outputs_from_headline"]:
    adapter_effect = adapter_effect[~adapter_effect["own_secret_leaked"]]

# Select each adapter's own target column and the opposite-word foil column.
adapter_effect["adapter_target_logit"] = adapter_effect.apply(
    lambda row: row[f"{row['condition']}_logit"], axis=1
)
adapter_effect["adapter_margin"] = adapter_effect.apply(
    lambda row: row[f"{row['condition']}_logit"]
    - row[f"{'blue' if row['condition'] == 'gold' else 'gold'}_logit"],
    axis=1,
)
adapter_effect["adapter_target_rank"] = adapter_effect.apply(
    lambda row: row[f"{row['condition']}_full_rank"], axis=1
)
base_long = pd.concat([
    base_effect.assign(
        target_word=target,
        base_target_logit=base_effect[f"{target}_logit"],
        base_margin=base_effect[f"{target}_logit"] - base_effect[f"{foil}_logit"],
        base_target_rank=base_effect[f"{target}_full_rank"],
    )
    for target, foil in (("gold", "blue"), ("blue", "gold"))
], ignore_index=True)
merged_effect = adapter_effect.merge(
    base_long[[
        "prompt_id", "method", "layer", "target_word", "base_target_logit",
        "base_margin", "base_target_rank",
    ]],
    left_on=["prompt_id", "method", "layer", "condition"],
    right_on=["prompt_id", "method", "layer", "target_word"],
    validate="many_to_one",
)
merged_effect["target_logit_lift"] = (
    merged_effect["adapter_target_logit"] - merged_effect["base_target_logit"]
)
merged_effect["target_margin_shift"] = (
    merged_effect["adapter_margin"] - merged_effect["base_margin"]
)
merged_effect["target_rr_lift"] = (
    1.0 / merged_effect["adapter_target_rank"]
    - 1.0 / merged_effect["base_target_rank"]
)
adapter_examples = (
    merged_effect.groupby(["prompt_id", "prompt_type", "condition", "method"], as_index=False)
    .agg(
        mean_target_logit_lift=("target_logit_lift", "mean"),
        mean_target_margin_shift=("target_margin_shift", "mean"),
        mean_target_rr_lift=("target_rr_lift", "mean"),
    )
)
adapter_summary = (
    adapter_examples.groupby(["condition", "prompt_type", "method"], as_index=False)
    .agg(
        n=("prompt_id", "size"),
        mean_target_logit_lift=("mean_target_logit_lift", "mean"),
        mean_target_margin_shift=("mean_target_margin_shift", "mean"),
        mean_target_rr_lift=("mean_target_rr_lift", "mean"),
    )
)
adapter_examples.to_csv(paths.result_dir / "adapter_vs_base_per_example.csv", index=False)
adapter_summary.to_csv(paths.result_dir / "adapter_vs_base_summary.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(
    data=adapter_examples, x="condition", y="mean_target_margin_shift",
    hue="method", errorbar=("ci", 95), seed=config["seed"], ax=ax,
)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Adapter-specific target margin shift relative to base")
saved_path = paths.figure_dir / "adapter_vs_base_margin_shift.png"
fig.savefig(saved_path, dpi=180, bbox_inches="tight")
display(fig)
display(adapter_summary)
display(adapter_examples)
print("Saved:", saved_path)


## 5. Layer trajectory at the last input token


In [ ]:
curve_columns = [
    "prompt_id", "condition", "target_word", "method", "layer",
    "position_roles_json", "own_secret_leaked", "target_full_rank",
]
curve_rows = pd.read_parquet(lens_path, columns=curve_columns)
curve_rows = curve_rows[curve_rows["target_word"].notna()].copy()
curve_rows["position_roles"] = curve_rows["position_roles_json"].map(json.loads)
curve_rows = curve_rows[
    curve_rows["position_roles"].map(lambda roles: "last_input" in roles)
]
if config["readout"]["exclude_leaking_outputs_from_headline"]:
    curve_rows = curve_rows[~curve_rows["own_secret_leaked"]]
curve_rows["reciprocal_rank"] = 1.0 / curve_rows["target_full_rank"]
layer_curve = (
    curve_rows.groupby(["condition", "method", "layer"], as_index=False)
    .agg(mean_reciprocal_rank=("reciprocal_rank", "mean"))
)
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=layer_curve, x="layer", y="mean_reciprocal_rank",
    hue="method", style="condition", ax=ax,
)
ax.axvspan(band_start, band_end, alpha=0.1, color="grey", label="prior band")
ax.set_title("Open-vocabulary MRR at the final input token")
saved_path = paths.figure_dir / "layer_curve_last_input_mrr.png"
fig.savefig(saved_path, dpi=180, bbox_inches="tight")
layer_curve.to_csv(paths.result_dir / "layer_curve_last_input_mrr.csv", index=False)
display(fig)
display(layer_curve.head())
print("Saved:", saved_path)


## 6. Gold/Blue candidate confusion

A useful target-specific pattern has Gold predicting Gold and Blue predicting
Blue. Consistent prediction of the same word in both rows indicates frequency
or readout bias rather than adapter-specific recovery.


In [ ]:
confusion_columns = [
    "prompt_id", "condition", "target_word", "method", "layer",
    "position_roles_json", "predicted_candidate", "own_secret_leaked",
]
confusion_rows = pd.read_parquet(lens_path, columns=confusion_columns)
confusion_rows = confusion_rows[confusion_rows["target_word"].notna()].copy()
confusion_rows["position_roles"] = confusion_rows["position_roles_json"].map(json.loads)
confusion_rows = confusion_rows[
    confusion_rows["position_roles"].map(lambda roles: "last_input" in roles)
    & confusion_rows["layer"].between(band_start, band_end)
]
if config["readout"]["exclude_leaking_outputs_from_headline"]:
    confusion_rows = confusion_rows[~confusion_rows["own_secret_leaked"]]
votes = (
    confusion_rows.groupby(["prompt_id", "condition", "method"])["predicted_candidate"]
    .agg(lambda values: values.value_counts().index[0])
    .rename("prediction")
    .reset_index()
)
confusion = pd.crosstab(
    [votes["method"], votes["condition"]], votes["prediction"], normalize="index"
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(confusion, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("Gold/Blue candidate prediction in the preregistered layer band")
saved_path = paths.figure_dir / "gold_blue_candidate_confusion.png"
fig.savefig(saved_path, dpi=180, bbox_inches="tight")
confusion.to_csv(paths.result_dir / "gold_blue_candidate_confusion.csv")
display(fig)
display(confusion)
print("Saved:", saved_path)


## 7. Exploratory layer × position heatmaps

These plots show target-minus-foil logit over the final input window and every
generated token. They are diagnostic, not confirmatory, because layer and
position are inspected exhaustively.


In [ ]:
HEATMAP_PROMPT = config["prompts"]["groups"]["lens_sweep"][0]
for condition in ("gold", "blue"):
    for method in ("logit_lens", "jlens"):
        heatmap_rows = pd.read_parquet(
            lens_path,
            columns=["prompt_id", "condition", "method", "layer", "position", "target_margin"],
        )
        selected_heatmap = heatmap_rows[
            heatmap_rows["prompt_id"].eq(HEATMAP_PROMPT)
            & heatmap_rows["condition"].eq(condition)
            & heatmap_rows["method"].eq(method)
        ]
        matrix = selected_heatmap.pivot(
            index="layer", columns="position", values="target_margin"
        )
        bound = float(np.nanpercentile(np.abs(matrix.to_numpy()), 98)) or 1.0
        fig, ax = plt.subplots(figsize=(15, 7))
        sns.heatmap(
            matrix, cmap="vlag", center=0, vmin=-bound, vmax=bound, ax=ax,
            cbar_kws={"label": "target logit - foil logit"},
        )
        ax.set_title(f"{method}: {condition} target margin - {HEATMAP_PROMPT}")
        filename = f"heatmap_{HEATMAP_PROMPT}_{condition}_{method}.png"
        saved_path = paths.figure_dir / filename
        fig.savefig(saved_path, dpi=180, bbox_inches="tight")
        matrix.to_csv(paths.result_dir / filename.replace(".png", ".csv"))
        display(fig)
        print("Saved:", saved_path)


## 8. Artifact inventory

Raw rendered prompts, token IDs and generations live under `data/raw_outputs`;
atomic lens cells under `artifacts/lens_outputs`; compact tables under
`results`; figures under `figures`. Keep the entire run ID together when
copying or archiving results.


In [ ]:
inventory = []
for root in (paths.raw_dir, paths.lens_dir, paths.result_dir, paths.figure_dir):
    for file in sorted(root.rglob("*")):
        if file.is_file():
            inventory.append({
                "path": str(file.relative_to(PROJECT_ROOT)),
                "size_mib": round(file.stat().st_size / 2**20, 3),
            })
inventory_frame = pd.DataFrame(inventory)
display(inventory_frame)
inventory_frame.to_csv(paths.result_dir / "artifact_inventory.csv", index=False)


## Interpretation checklist

- J-Lens beating Logit Lens only in earlier layers supports added value from
  the Jacobian transport.
- Both methods succeeding only late suggests little advantage over Logit Lens.
- Signal before generation and specific to the correct adapter is stronger
  evidence than signal appearing only after topical hints.
- Signal on direct-refusal prompts may be weak because refusal need not retrieve
  the secret.
- Literal output leakage invalidates hidden-secret evidence for that rollout.
- None of these readouts establishes causal use; they establish decodability
  under the named method, layer, and position.
